# U11_C1 — Comparing Two Genomes
**Course:** Concepts of Bioinformatics (CBBI-501) · **Unit 11:** Comparative Genome Analysis
**Type:** Course notebook · **Time:** 40 min · **Difficulty:** ★★☆
**Goes with:** Lectures L29–L30 (Genome Annotation & Comparative Genomics)
**Before this:** U10_C1

## 1. What you will learn
- **Compare** fundamental genome statistics (length, GC content, gene count) across related genomes [LO8].
- **Calculate** coding density and recognize overlapping reading frames in compact viral genomes [LO8].
- **Identify** homologous gene sets from tabular BLAST comparisons [LO8].
- **Evaluate** gene order conservation (synteny) between divergent viral genomes [LO8].


## 2. Biological question
Bacteriophages $\phi X174$ (5386 bp) and G4 (5577 bp) are single-stranded DNA microviruses that infect enteric bacteria. They were the very first DNA genomes ever sequenced (Fred Sanger, 1977). Despite substantial nucleotide divergence, how do their gene repertoires compare? How do compact viruses squeeze 11 proteins into only 5.4 kb of DNA, and how does comparative genomics prove their functional orthology?


In [ ]:
import sys, subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "biopython"], check=True)

NOTEBOOK_ID = "U11_C1"
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data").is_dir())
DATA_DIR = ROOT / "data"
OUT_DIR = ROOT / "outputs" / NOTEBOOK_ID
OUT_DIR.mkdir(parents=True, exist_ok=True)

import Bio
print("Python", sys.version.split()[0], "| Biopython", Bio.__version__)

## 3. Concept 1: Fundamental Genome Metrics
### Theory
Comparative genomics begins by contrasting summary macro-metrics:
1. **Genome Size**: Measured in base pairs (bp), kilobases (kb), or megabases (Mb).
2. **Overall GC Percentage**: Dictated by mutational bias, replication machinery, and ecological niche.
3. **Gene and CDS Count**: Number of transcribed units and protein-coding open reading frames.


In [ ]:
# Worked Example: Loading and comparing PhiX174 vs G4 Phage
from Bio import SeqIO

phix_path = DATA_DIR / "genomes" / "phix174.gb"
g4_path = DATA_DIR / "genomes" / "g4_phage.gb"

phix_rec = SeqIO.read(phix_path, "genbank")
g4_rec = SeqIO.read(g4_path, "genbank")

def summarize_genome(rec):
    length = len(rec.seq)
    gc = ((rec.seq.count("G") + rec.seq.count("C")) / length) * 100
    cds_features = [f for f in rec.features if f.type == "CDS"]
    return {
        "accession": rec.id,
        "organism": rec.annotations.get("organism", ""),
        "length_bp": length,
        "gc_percent": round(gc, 2),
        "cds_count": len(cds_features)
    }

meta_phix = summarize_genome(phix_rec)
meta_g4 = summarize_genome(g4_rec)

print("=== Comparative Summary Table ===")
print(f"{'Feature':20} | {'PhiX174':15} | {'G4 Phage':15}")
print("-" * 56)
print(f"{'Accession':20} | {meta_phix['accession']:15} | {meta_g4['accession']:15}")
print(f"{'Genome Length':20} | {meta_phix['length_bp']:<15} | {meta_g4['length_bp']:<15}")
print(f"{'GC Content (%)':20} | {meta_phix['gc_percent']:<15} | {meta_g4['gc_percent']:<15}")
print(f"{'Protein CDS Count':20} | {meta_phix['cds_count']:<15} | {meta_g4['cds_count']:<15}")


## 4. Concept 2: Coding Density and Overlapping Reading Frames
### Theory
**Coding density** measures the proportion of genomic DNA dedicated to protein coding:
$$\text{Coding Density} = \frac{\sum \text{Length of all CDS (bp)}}{\text{Genome Length (bp)}} \times 100$$
- In humans, protein-coding exons comprise $< 2\%$ of the genome.
- In bacteria, coding density is $\approx 85-90\%$.
- In microviruses (like $\phi X174$), coding density can exceed $100\%$ because genes overlap and are translated in different reading frames (e.g. gene B is encoded completely within gene A; gene E is within gene D).


In [ ]:
# Worked Example: Calculating coding density
def calc_coding_density(rec):
    total_cds_nt = 0
    for f in rec.features:
        if f.type == "CDS":
            total_cds_nt += len(f.extract(rec.seq))
    density = (total_cds_nt / len(rec.seq)) * 100
    return total_cds_nt, round(density, 1)

nt_phix, dens_phix = calc_coding_density(phix_rec)
nt_g4, dens_g4 = calc_coding_density(g4_rec)

print(f"PhiX174: {nt_phix} coding nt in {len(phix_rec.seq)} bp genome -> Coding Density = {dens_phix}%")
print(f"G4 Phage: {nt_g4} coding nt in {len(g4_rec.seq)} bp genome -> Coding Density = {dens_g4}%")


## 5. Concept 3: Shared Homologous Genes via Tabular BLAST
### Theory
Comparing gene content involves running all-against-all protein BLAST (BLASTP) between the proteomes of two genomes. The resulting High-Scoring Segment Pairs identify homologous pairs, their percent identity, and evolutionary divergence.


In [ ]:
# Worked Example: Inspecting shared orthologues from BLAST table
import pandas as pd

blast_path = DATA_DIR / "expected" / "blast_phix174_vs_g4.tsv"
df_shared = pd.read_csv(blast_path, sep="\t")

print("=== Homologous Proteins Shared Between PhiX174 and G4 Phage ===")
print(df_shared[["query_id", "subject_id", "identity", "evalue", "subject_title"]])

avg_ident = df_shared["identity"].mean()
print(f"\nMean sequence identity across all shared viral proteins: {avg_ident:.1f}%")


## 6. Concept 4: Conserved Gene Order (Synteny)
### Theory
**Synteny** refers to the preservation of gene order on corresponding chromosomes between different species. In bacteriophages, the capsid, spike, replication, and lysis genes occur in a strictly conserved sequential order, demonstrating strong evolutionary constraint on transcriptional timing.


In [ ]:
# Worked Example: Verifying collinear synteny
print("Ordered gene products in G4 matching PhiX174 query order:")
for idx, title in enumerate(df_shared["subject_title"], 1):
    print(f"  Gene {idx:2}: {title}")


## 7. Summary
- Comparative genomics reveals conserved core genes and structural genome reorganization.
- Viral genomes achieve extreme coding density ($>90-100\%$) through overlapping reading frames.
- Synteny analysis confirms that gene order is conserved even when sequence identity has decayed to $\sim 60\%$.


## 8. Next Steps
- Compare polyomaviruses (SV40 vs BKV) in `U11_P_Comparative_Genomics.ipynb`.
- Self-test comparative genomics concepts in `U11_Q_Comparative_Genomics.ipynb`.
